In [0]:
%pip install langdetect FlagEmbedding databricks-ai-search --quiet

In [0]:
%pip install langdetect FlagEmbedding --quiet

In [0]:
import re
import pandas as pd
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window
import sklearn
import scipy
import unicodedata 
import langdetect

# ---- Layer ----
CATALOG = "us_gmsgq_dev"
ALYT    = "gms_us_alyt"     
MART    = "gms_us_mart"

# ---- RAW reference inputs ----
RAW_CLINICAL = f"{CATALOG}.{ALYT}.Clinicalid_deviations"
RAW_DOCS = f"{CATALOG}.{ALYT}.documents_number_deviations"
RAW_ACRONYM  = f"{CATALOG}.{ALYT}.Acronyms_other_deviations"
RAW_CRO      = f"{CATALOG}.{MART}.gmsgq_veeva_documents" #replaced Cro_deviations
RAW_DEVICE   = f"{CATALOG}.{ALYT}.rd_device_list_deviations"
RAW_VENDORS = f"{CATALOG}.{ALYT}.vendors_deviations"

# ---- SOURCE deviation data (read-only, in mart) ----
SOURCE_TABLE = f"{CATALOG}.{MART}.tw_deviation_data_formatted_rdq"

# ---- OUTPUT lookups ----
REF_CLINICAL = f"{CATALOG}.{ALYT}.ref_clinical_norm"
REF_DOCS     = f"{CATALOG}.{ALYT}.ref_docs_norm"
REF_ACRONYM  = f"{CATALOG}.{ALYT}.ref_acronym_norm"
REF_CRO      = f"{CATALOG}.{ALYT}.ref_cro_norm"
REF_DEVICE   = f"{CATALOG}.{ALYT}.ref_device_norm"
REF_VENDORS = f"{CATALOG}.{ALYT}.ref_vendors_norm"
REF_UNIFIED  = f"{CATALOG}.{ALYT}.ref_glossary_unified"

# ---- The contract every lookup MUST expose (extra cols allowed) ----
LOOKUP_SCHEMA = ["key_norm", "entity_type", "canonical_id", "enrichment_text"]

print("Config loaded.")
print("  Inputs :", RAW_CLINICAL, RAW_DOCS, RAW_ACRONYM, RAW_CRO, RAW_DEVICE, RAW_VENDORS, sep="\n           ")
print("  Source :", SOURCE_TABLE)
print("  Output :", REF_UNIFIED)

In [0]:
# ============================================================================
# SECTION A — CLINICAL IDs  (extract + parse; 3-source logic: free text + protocol + program)
# ============================================================================
CLINICAL_ID_ALTERNATIVES = [
    r"TAK[-\s_]?\d{2,4}[-_/]\d{3,4}",
    r"TAK[-\s_]?\d{2,4}",
    r"MLN[-\s_]?\d{3,4}[-_/]CCT-\d{2,4}",
    r"MLN[-\s_]?\d{3,4}[-_/]\d{2,4}",
    r"MLN[-\s_]?\d{3,4}",
    r"SHP[-\s_]?\d{3,4}[-_/]\d{2,4}",
    r"SHP[-\s_]?\d{3,4}",
    r"HGT[-\s_]?[A-Z]{2,4}[-_/]\d{2,4}",
    r"CCT[-_]?\d{2,4}",
    r"DEN[-_]?\d{2,4}",
    r"C\d{5}",
]
ID_REGEX = re.compile("|".join(CLINICAL_ID_ALTERNATIVES), flags=re.IGNORECASE)

# Program_Number filter: keep only values whose leading token is a valid clinical-ID shape.
PROGRAM_FILTER_STR = (
    r"^(TAK-?\d{2,4}|MLN\d{4}|SHP-?\d{3,4}|HGT-[A-Z]{2,4}-\d{2,4}"
    r"|C\d{5}|CCT-\d{2,4}|DEN-\d{2,4})"
)

def _which_prefix(u):
    for p in ("TAK", "MLN", "SHP", "HGT", "CCT", "DEN"):
        if u.startswith(p):
            return p
    if re.match(r"C\d{5}$", u):
        return "INTERNAL"
    return "UNKNOWN"

def parse_clinical_id(raw):
    u = re.sub(r"-{2,}", "-", re.sub(r"[\s_/]+", "-", str(raw).upper())).strip("-")
    prefix = _which_prefix(u)
    compound = suffix = None
    if prefix in ("TAK", "MLN", "SHP"):
        m = re.match(prefix + r"-?(\d+)(?:-(.+))?$", u)
        if m: compound, suffix = m.group(1), m.group(2)
        key = (prefix + "-" + compound) if compound else u
    elif prefix == "HGT":
        m = re.match(r"HGT-([A-Z]{2,4})-(\d+)$", u)
        if m: compound, suffix = m.group(1), m.group(2)
        key = ("HGT-" + compound) if compound else u
    elif prefix in ("CCT", "DEN"):
        m = re.match(prefix + r"-?(\d+)$", u)
        compound, suffix, key = prefix, (m.group(1) if m else None), u
    elif prefix == "INTERNAL":
        m = re.match(r"C(\d+)$", u)
        compound, key = (m.group(1) if m else None), u
    else:
        key = u
    return {"raw": raw, "canonical": u, "prefix": prefix,
            "compound_number": compound, "study_suffix": suffix, "normalized_key": key}

@F.udf(T.ArrayType(T.StringType()))
def extract_clinical_keys_udf(text):
    """(1) Free text: extract clinical IDs, return normalized compound+study keys."""
    if not text: return []
    keys = set()
    for m in ID_REGEX.finditer(str(text)):
        p = parse_clinical_id(m.group(0))
        if p["normalized_key"]: keys.add(p["normalized_key"])
        if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

@F.udf(T.ArrayType(T.StringType()))
def protocol_keys_udf(v):
    """(2) Study_Protocol: split on ';', TRUST ALL values -> normalized clinical keys."""
    if not v: return []
    _NA_PATTERNS = {"N/A", "NA", "N-A", "NONE", "NULL", "-", "--", ""}
    keys = set()
    for s in re.split(r"\s*;\s*", str(v)):
        s = s.strip()
        if not s or s.upper() in _NA_PATTERNS: continue
        p = parse_clinical_id(s)
        if p["normalized_key"]: keys.add(p["normalized_key"])
        if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

@F.udf(T.ArrayType(T.StringType()))
def program_keys_udf(v):
    """(3) Program_Number: split on ';', keep only valid clinical-ID patterns."""
    if not v: return []
    keys = set()
    for s in re.split(r"\s*;\s*", str(v)):
        s = s.strip()
        if s and re.match(PROGRAM_FILTER_STR, s):
            p = parse_clinical_id(s)
            if p["normalized_key"]: keys.add(p["normalized_key"])
            if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

def _strip_accents(s):
    """Fold accented Latin chars to ASCII (Genève -> Geneve). Non-Latin left as-is."""
    nfkd = unicodedata.normalize("NFKD", s)
    return "".join(c for c in nfkd if not unicodedata.combining(c))

def clean_freetext(v):
    """Case-PRESERVING cleanup: unicode hyphens/quotes/whitespace + accent folding.
       Do NOT lowercase — acronym & CRO extraction rely on original casing."""
    if not v:
        return None
    s = unicodedata.normalize("NFKC", str(v))
    s = re.sub(r"[\u2010-\u2015\u2212]", "-", s)   # unicode hyphens/minus -> '-'
    s = s.replace("\u00a0", " ")                    # non-breaking space -> space
    s = _strip_accents(s)                           # fold accents for matching
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip() or None

clean_freetext_udf = F.udf(clean_freetext, T.StringType())

# --- Language detection (English-only embedding model requires English text) ---
def _detect_lang(text):
    """Return ISO code ('en', 'de', ...) or None. Short/ID-only ASCII text -> 'en'.
       Text containing CJK / substantial non-Latin script is NOT auto-whitelisted."""
    if not text or not text.strip():
        return None
    stripped = text.strip()

    # If there's any CJK / non-Latin letters, fall through to real detection.
    has_cjk = re.search(
        r"[\u3040-\u30ff\u3400-\u4dbf\u4e00-\u9fff\uac00-\ud7af]", stripped
    )

    if not has_cjk:
        # ASCII-ish: allow the code-only escape hatch
        residual = re.sub(r"\b[A-Z]{2,}[-\s]?\d+\b", " ", stripped)
        if not re.search(r"[A-Za-z]{3,}", residual):
            return "en"

    try:
        from langdetect import detect, DetectorFactory
        DetectorFactory.seed = 0
        return detect(stripped)
    except Exception:
        return None

def keep_if_english(text):
    """Return the text if English (or code-only), else None. Case-preserving."""
    lang = _detect_lang(text)
    return text if lang == "en" else None

keep_if_english_udf = F.udf(keep_if_english, T.StringType())

# ============================================================================
# SECTION B — DOCUMENTS  (regex EXACTLY matching SQL patterns)
# ============================================================================
DOC_PATTERNS = [
    r"\bSOP-\d+",
    r"\bSPEC-\d+",
    r"\bMTHD-\d+",
    r"\bMTD-\d+",      # legacy method prefix seen in your reference (MTD-002598)
    r"\bPROC-\d+",
    r"\bTOOL-\d+",
    r"\bFORM-\d+",
    r"\bWI-\d+",
]
DOC_REGEX = re.compile("|".join(DOC_PATTERNS), flags=re.IGNORECASE)
DOC_PREFIXES = r"(SOP|SPEC|MTHD|MTD|PROC|TOOL|FORM|WI)"

def norm_doc(v):
    """Canonical doc key. Note: MTD->MTHD collapse so legacy+current align."""
    if not v: return None
    u = re.sub(r"-{2,}", "-", re.sub(r"[\s_]+", "-", str(v).upper())).strip("-")
    if not re.match(DOC_PREFIXES + r"-?\d", u): return None
    u = re.sub(r"^MTD-", "MTHD-", u)          # unify legacy method prefix
    return u

norm_doc_udf = F.udf(norm_doc, T.StringType())

@F.udf(T.ArrayType(T.StringType()))
def extract_doc_keys_udf(text):
    """Extraction-side twin of your SQL: pull all doc IDs from text, normalized."""
    if not text: return []
    out = set()
    for m in DOC_REGEX.finditer(str(text)):
        k = norm_doc(m.group(0))
        if k: out.add(k)
    return list(out)

@F.udf(T.ArrayType(T.StringType()))
def legacy_doc_keys_udf(v):
    """Reference-side: parse messy legacy_numbers__c like
       'LSHIRE_1174436_6_0;;n/a;;TO SOP-0895' -> ['SOP-0895']."""
    if not v: return []
    out = set()
    for chunk in str(v).split(";;"):
        chunk = chunk.strip()
        if not chunk or chunk.lower() == "n/a": continue
        for m in re.finditer(DOC_PREFIXES + r"[-\s]?\d{3,7}", chunk, re.I):
            k = norm_doc(m.group(0))
            if k: out.add(k)
    return list(out)

# ============================================================================
# SECTION C — ACRONYMS  (regex candidate extraction + KNOWN-SET filter)
# ============================================================================
ACRONYM_CANDIDATE = re.compile(r"\(?[A-Z]{2,}\)?(?:[-/][A-Z0-9]+)?|\([A-Z]{2,}\)\s?[A-Z]{2,}")
ACR_DEF_JUNK = {"", "n/a", "na", "none", "null", "somebody", "unknown",
                "tbd", "todo", "test", "xxx", "-", "--", "."}
# ... in the ref_acronym chain, after .filter(F.col("Category").isin("Medical","Industry")):

def norm_acr(v):
    if not v: return None
    u = re.sub(r"\s+", " ", str(v).strip()).upper()
    return u or None

norm_acr_udf = F.udf(norm_acr, T.StringType())

def make_extract_acronym_udf(known_keys):
    """Factory: returns a UDF that extracts only acronyms present in the known set.
       Min length 3 applied here to suppress short false-positive matches (e.g. 'IN', 'OF')."""
    known = frozenset(k for k in known_keys if k and len(k) >= 3)
    @F.udf(T.ArrayType(T.StringType()))
    def _udf(text):
        if not text: return []
        out = set()
        for m in ACRONYM_CANDIDATE.finditer(str(text)):
            k = norm_acr(m.group(0))
            if k and k in known:
                out.add(k)
        return list(out)
    return _udf

# ============================================================================
# SECTION D — CRO / SYSTEMS
# ============================================================================
CRO_DESC_JUNK = {"", "1", "n/a", "na", "none", "null", "-", "--", ".",
                 "broken", "tbd", "todo", "unknown", "test", "xxx"}

def norm_name(v):
    if not v: return None
    u = re.sub(r"[^A-Z0-9 ]", " ", str(v).upper())
    u = re.sub(r"\s+", " ", u).strip()
    if not u: return None
    # Real org names have letters; require >=2 alphabetic chars.
    # "2020-10-26 15:40:45" -> "2020 10 26 15 40 45" -> 0 letters -> drop
    if len(re.findall(r"[A-Z]", u)) < 2:
        return None
    return u

norm_name_udf = F.udf(norm_name, T.StringType())


@F.udf(T.ArrayType(T.StringType()))
def cro_variants_udf(name, disp):
    """Reference-side: normalized name/displayName variants for the CRO pool (alias excluded)."""
    return list({norm_name(x) for x in (name, disp) if x and norm_name(x)})

def make_extract_cro_exact_udf(known_keys):
    """Exact-match twin."""
    known = frozenset(known_keys)
    _cand = re.compile(r"\b([A-Z][A-Za-z0-9]+(?:\s+[A-Z][A-Za-z0-9]+){0,3}|[A-Z]{2,})\b")
    @F.udf(T.ArrayType(T.StringType()))
    def _udf(text):
        if not text: return []
        out = set()
        for m in _cand.finditer(str(text)):
            k = norm_name(m.group(1))
            if k and k in known:
                out.add(k)
        return list(out)
    return _udf


# ============================================================================
# SECTION E — DEVICES  (substring match on nonsensical names; context = type + owner)
# ============================================================================
# Business-owner abbreviation expansions (longest keys first at match time)
BUSINESS_OWNER_MAP = {
    "RGH TAU": "Rare Genetics and Hematology Therapeutic Area Unit",
    "GI TAU":  "Gastrointestinal Therapeutic Area Unit",
    "OTAU":    "Oncology Therapeutic Area Unit",
    "PDT":     "Plasma Derived Therapy",
}

def expand_business_owner(v):
    """Expand known business-owner abbreviations found anywhere in the value."""
    if v is None or str(v).strip() == "":
        return ""
    s = str(v)
    # replace longer keys first to avoid partial shadowing
    for abbr in sorted(BUSINESS_OWNER_MAP, key=len, reverse=True):
        s = re.sub(rf"\b{re.escape(abbr)}\b", BUSINESS_OWNER_MAP[abbr], s, flags=re.I)
    return s
expand_business_owner_udf = F.udf(expand_business_owner, T.StringType())

def norm_device(v):
    """Lowercase + collapse whitespace. Device names are matched as substrings,
       so we keep them permissive (no aggressive stripping)."""
    if not v:
        return None
    u = re.sub(r"\s+", " ", str(v)).strip().lower()
    return u or None

norm_device_udf = F.udf(norm_device, T.StringType())

@F.udf(T.ArrayType(T.StringType()))
def device_variants_udf(name, alias, long_name):
    """Reference-side: normalized name/alias/long_name variants for the device pool."""
    return list({norm_device(x) for x in (name, alias, long_name) if x and norm_device(x)})

def make_extract_device_exact_udf(known_keys):
    """Substring matcher: return every device key that appears in the (lowercased) text.
       Mirrors the SQL `instr(lower(text), lower(name)) > 0` logic."""
    known = list(frozenset(k for k in known_keys if k and len(k) >= 4))  # skip 1-3 char noise
    @F.udf(T.ArrayType(T.StringType()))
    def _udf(text):
        if not text:
            return []
        low = str(text).lower()
        return [k for k in known if k in low]
    return _udf

# ============================================================================
# SECTION F — VENDORS  (substring match; context = vendor name + country)
# ============================================================================
def norm_vendor(v):
    """Lowercase + collapse whitespace. Vendor names are matched as substrings,
       so we keep them permissive (no aggressive stripping)."""
    if not v:
        return None
    u = re.sub(r"\s+", " ", str(v)).strip().lower()
    return u or None

norm_vendor_udf = F.udf(norm_vendor, T.StringType())

@F.udf(T.ArrayType(T.StringType()))
def vendor_variants_udf(nm):
    """Reference-side: normalized nm variant for the vendor pool.
       vendors_deviations exposes a single name column (nm); no alias or long_name."""
    return list({norm_vendor(x) for x in (nm,) if x and norm_vendor(x)})

@F.udf(T.ArrayType(T.StringType()))
def text_ngrams_udf(text):
    """Generate overlapping 1-to-5-word n-grams from text for vendor broadcast JOIN.
       Each n-gram is equi-joined against REF_VENDORS.key_norm in the src_keys cell."""
    if not text:
        return []
    words = re.sub(r"\s+", " ", str(text).lower()).strip().split()
    out = set()
    for n in range(1, 6):
        for i in range(len(words) - n + 1):
            gram = " ".join(words[i:i + n])
            if len(gram) >= 4:
                out.add(gram)
    return list(out)

def make_extract_vendor_exact_udf(known_keys):
    """Substring matcher: return every vendor key that appears in the (lowercased) text.
       Mirrors the device pattern — min length 4 to suppress noise."""
    known = list(frozenset(k for k in known_keys if k and len(k) >= 4))
    @F.udf(T.ArrayType(T.StringType()))
    def _udf(text):
        if not text:
            return []
        low = str(text).lower()
        return [k for k in known if k in low]
    return _udf

print("Shared UDFs registered: clinical (3-source), doc, acronym(factory), cro (exact+fuzzy), device, vendor.")

In [0]:
ac = spark.table(RAW_ACRONYM)

# Guard: don't let short acronyms shadow real clinical/doc IDs
ID_LIKE = r"^(TAK|MLN|SHP|HGT|CCT|DEN|SOP|SPEC|MTHD|FORM|TOOL|WI|PROC)[-\s]?\d"

def _britfix(col):
    """Normalise British -ise/-isation spellings → American -ize/-ization.
       Applied before deduplication so near-identical spellings collapse to one."""
    for pat, rep in [('isation','ization'), (r'ised\b','ized'), (r'ising\b','izing'), (r'ise\b','ize')]:
        col = F.regexp_replace(col, pat, rep)
    return col

ref_acronym = (
    ac.withColumn("key_norm", norm_acr_udf(F.col("Acronym")))
      .filter(F.col("key_norm").isNotNull())
      .filter(F.length("key_norm") >= 3)                      # drop 1-2 char noise
      .filter(~F.col("key_norm").rlike(ID_LIKE))              # don't shadow IDs
      .filter(F.col("Category").isin("Medical", "Industry")).filter(F.col("Definition").isNotNull())
      .filter(~F.lower(F.trim(F.col("Definition"))).isin(*ACR_DEF_JUNK))
      .filter(F.length(F.trim(F.col("Definition"))) >= 2) 
      .groupBy("key_norm")
      .agg(
          F.collect_set(_britfix(F.lower(F.trim(F.col("Definition"))))).alias("definitions"),
          F.first("Acronym", ignorenulls=True).alias("canonical_id"),
      )
      .withColumn("entity_type", F.lit("ACRONYM"))
      .withColumn("enrichment_text", F.concat(
          F.lit("[ACRONYM] \""), F.col("key_norm"), F.lit("\"\n"),
          F.lit("Canonical: "),   F.col("canonical_id"), F.lit("\n"),
          F.lit("Definitions: {"),
          F.array_join(F.array_sort(F.col("definitions")), ", "),
          F.lit("}")))
      .select(*LOOKUP_SCHEMA)
)

(ref_acronym.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_ACRONYM))
spark.sql(f"COMMENT ON TABLE {REF_ACRONYM} IS "
          "'Sense-aware acronym lookup: key_norm→all definitions. Built from raw_acronym.'")

display(spark.table(REF_ACRONYM).limit(20))

# Build a plain dict {ACRONYM_UPPER: first_definition} for TA/modality expansion in Cell 4
_acr_pd = (ac.filter(F.col("Category").isin("Medical", "Industry"))
             .select("Acronym", "Definition").toPandas())
ACR_MAP = {norm_acr(a): d for a, d in zip(_acr_pd["Acronym"], _acr_pd["Definition"]) if norm_acr(a)}

# Hand-curated TA overrides win over the generic acronym table
TA_OVERRIDES = {
    "NS": "Neuroscience", "ONC": "Oncology", "GI": "Gastroenterology",
    "RARE": "Rare Diseases", "PDT": "Plasma-Derived Therapies",
}
def expand_abbr(v):
    if v is None or str(v).strip() == "":
        return ""
    u = norm_acr(v)
    return TA_OVERRIDES.get(u) or ACR_MAP.get(u) or v   # fall back to original text
expand_abbr_udf = F.udf(expand_abbr, T.StringType())

print(f"Acronym map size: {len(ACR_MAP):,}")

In [0]:

acr_key_set = set(r["key_norm"] for r in spark.table(REF_ACRONYM).select("key_norm").collect())
extract_acronym_keys_udf = make_extract_acronym_udf(acr_key_set)   # pass the plain set
print(f"Acronym known-set size: {len(acr_key_set):,}")

In [0]:
# ============================================================================
# CELL 5 — REF_CLINICAL  (match on ALL alias cols; rich expanded context)
# ============================================================================
clin = spark.table(RAW_CLINICAL)
print("RAW_CLINICAL columns:", clin.columns)   # <- verify alias/context names

# --- Alias columns: any of these, if present in text, should resolve the row ---
CLIN_ALIAS_COLS = [
    "Parent_Node", "Name", "Protocol Number", "Alternate_Name",
    "Development_Name", "Parent_Alias", "Grand_Parent", "Grandparent_Alias",
]

def _clin_c(name):
    return F.col(f"`{name}`") if name in clin.columns else F.lit(None).cast("string")

# UDF: build normalized keys from every available alias value
@F.udf(T.ArrayType(T.StringType()))
def clinical_alias_keys_udf(*vals):
    keys = set()
    for v in vals:
        if v and str(v).strip():
            p = parse_clinical_id(str(v))
            if p["normalized_key"]: keys.add(p["normalized_key"])
            if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

# --- Context (expand TA/modality abbreviations) ---
clin_enriched = clin.withColumn(
    "enrichment_text",
    F.concat_ws(
        "\n",
        F.concat(F.lit("Generic Name: "),      F.coalesce(_clin_c("Generic_Name"), F.lit(""))),
        F.concat(F.lit("Modality: "),         expand_abbr_udf(_clin_c("Modality"))),
        F.concat(F.lit("Therapeutic Area: "), expand_abbr_udf(_clin_c("PF_TherapeuticArea"))),
        F.concat(F.lit("Finance TA Grouping: "), expand_abbr_udf(_clin_c("finance_TA_Grouping"))),
        F.concat(F.lit("Indication: "),       F.coalesce(_clin_c("IND_DESC"), F.lit(""))),
        F.concat(F.lit("Target: "),           F.coalesce(_clin_c("Target_Long_Name"), F.lit(""))),
        F.concat(F.lit("Mechanism: "),        F.coalesce(_clin_c("Mechanism"), F.lit(""))),
    ),
).withColumn(
    "canonical_id",
    F.coalesce(_clin_c("Development_Name"), _clin_c("Name"), _clin_c("Protocol Number")),
)

alias_cols_present = [c for c in CLIN_ALIAS_COLS if c in clin.columns]
ref_clinical = (
    clin_enriched
    .withColumn("key_norm", F.explode(
        clinical_alias_keys_udf(*[F.col(f"`{c}`") for c in alias_cols_present])))
    .filter(F.col("key_norm").isNotNull() & (F.col("key_norm") != ""))
    .withColumn("entity_type", F.lit("CLINICAL_ID"))
    .withColumn("enrichment_text", F.concat(
        F.lit("[CLINICAL_ID] \""), F.col("key_norm"), F.lit("\"\n"),
        F.lit("Canonical: "),      F.col("canonical_id"), F.lit("\n"),
        F.col("enrichment_text")
    ))
    .select(*LOOKUP_SCHEMA)
    .dropDuplicates(["key_norm"])
)

(ref_clinical.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_CLINICAL))
spark.sql(f"COMMENT ON TABLE {REF_CLINICAL} IS "
          "'Clinical-ID lookup keyed on all alias columns; context expanded via acronym map.'")
display(spark.table(REF_CLINICAL).limit(20))

In [0]:
# ============================================================================
# CELL 6 — REF_DOCS  (keys from doc# + previous# + legacy; context = title)
# ============================================================================
docs = spark.table(RAW_DOCS)
print("RAW_DOCS columns:", docs.columns)   # <- verify names below

DOC_NUM_COL    = "document_number__v"
DOC_PREV_COL   = "previous_document_number__c"
DOC_LEGACY_COL = "legacy_numbers__c"
DOC_TITLE_COL  = "title__v"

def _dc(name):
    return F.col(f"`{name}`") if name in docs.columns else F.lit(None).cast("string")

docs_enriched = docs.withColumn(
    "enrichment_text",
    F.concat(F.lit("Document Title: "), F.coalesce(_dc(DOC_TITLE_COL), F.lit(""))),
).withColumn("canonical_id", _dc(DOC_NUM_COL))

# straightforward single-value doc columns -> norm_doc
def _doc_keys_from(col_name):
    if col_name not in docs.columns:
        return None
    return (docs_enriched
            .withColumn("key_norm", norm_doc_udf(F.col(f"`{col_name}`")))
            .filter(F.col("key_norm").isNotNull()))

parts = [df for df in (_doc_keys_from(DOC_NUM_COL), _doc_keys_from(DOC_PREV_COL)) if df is not None]

# messy legacy column -> legacy_doc_keys_udf (explode)
if DOC_LEGACY_COL in docs.columns:
    parts.append(
        docs_enriched
        .withColumn("key_norm", F.explode(legacy_doc_keys_udf(F.col(f"`{DOC_LEGACY_COL}`"))))
        .filter(F.col("key_norm").isNotNull() & (F.col("key_norm") != ""))
    )

ref_docs_union = parts[0]
for df in parts[1:]:
    ref_docs_union = ref_docs_union.unionByName(df)

ref_docs = (
    ref_docs_union
    .withColumn("entity_type", F.lit("DOCUMENT"))
    .withColumn("enrichment_text", F.concat(
        F.lit("[DOCUMENT] \""), F.col("key_norm"), F.lit("\"\n"),
        F.lit("Canonical: "),   F.col("canonical_id"), F.lit("\n"),
        F.col("enrichment_text")
    ))
    .select(*LOOKUP_SCHEMA)
    .withColumn("_len", F.length("enrichment_text"))
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("key_norm").orderBy(F.col("_len").desc())))
    .filter(F.col("_rn") == 1)
    .select(*LOOKUP_SCHEMA)
)

(ref_docs.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_DOCS))
spark.sql(f"COMMENT ON TABLE {REF_DOCS} IS "
          "'Doc lookup keyed on document_number__v + previous + legacy; context = title__v.'")
display(spark.table(REF_DOCS).limit(20))

In [0]:
# ============================================================================
# CELL 7 — REF_CRO  (normalized name pool; context = description)
# ============================================================================
cro = spark.table(RAW_CRO)
CRO_NAME_COL  = "name"
CRO_DISP_COL  = "displayName"
CRO_DESC_COL  = "description"

def _cc(name):
    return F.col(f"`{name}`") if name in cro.columns else F.lit(None).cast("string")


desc_clean = F.when(
    F.trim(F.coalesce(_cc(CRO_DESC_COL), F.lit(""))).isin("", "1", "n/a", "N/A"),
    F.lit(None)
).otherwise(_cc(CRO_DESC_COL))

cro_enriched = (
    cro
    # Drop rows where either name OR displayName is null, a datetime, or a description paragraph
    .filter(_cc(CRO_NAME_COL).isNotNull() & _cc(CRO_DISP_COL).isNotNull())
    .filter(
        ~_cc(CRO_NAME_COL).rlike(r'^\d{4}-\d{2}-\d{2}') &
        ~_cc(CRO_DISP_COL).rlike(r'^\d{4}-\d{2}-\d{2}')
    )
    .filter((F.length(_cc(CRO_NAME_COL)) <= 80) & (F.length(_cc(CRO_DISP_COL)) <= 80))
    .withColumn("desc_clean", desc_clean)
    .filter(F.col("desc_clean").isNotNull())
    .withColumn(
        "enrichment_text",
        F.concat_ws(
            "\n",
            F.concat(F.lit("CRO / System Name: "),
                     F.coalesce(_cc(CRO_DISP_COL), _cc(CRO_NAME_COL), F.lit(""))),
            F.concat(F.lit("Description: "), F.col("desc_clean")),
        ),
    )
    .withColumn("canonical_id", _cc(CRO_NAME_COL))
)

ref_cro = (
    cro_enriched
    .withColumn("key_norm", F.explode(
        cro_variants_udf(_cc(CRO_NAME_COL), _cc(CRO_DISP_COL))))
    .filter(F.col("key_norm").isNotNull() & (F.col("key_norm") != "") & (F.length("key_norm") >= 4) & (F.length("key_norm") <= 50))
    .withColumn("entity_type", F.lit("CRO"))
    .withColumn("enrichment_text", F.concat(
        F.lit("[CRO] \""), F.col("key_norm"), F.lit("\"\n"),
        F.lit("Canonical: "), F.col("canonical_id"), F.lit("\n"),
        F.col("enrichment_text")
    ))
    .select(*LOOKUP_SCHEMA)
    .dropDuplicates(["key_norm"])
)


(ref_cro.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_CRO))
spark.sql(f"COMMENT ON TABLE {REF_CRO} IS "
          "'CRO/system lookup: normalized name/display/alias variants; context = description (\\'1\\'/n/a flagged as missing).'")

# CRO extraction uses a broadcast JOIN on n-grams in the src_keys cell — no factory UDF needed.
print(f"CRO pool size: {spark.table(REF_CRO).count():,}")

display(spark.table(REF_CRO).limit(20))

In [0]:
# ============================================================================
# CELL 7b — REF_DEVICE  (substring name pool; context = Device_Type + business_owner)
# ============================================================================
dev = spark.table(RAW_DEVICE)
print("RAW_DEVICE columns:", dev.columns)   # <- verify names below

DEV_NAME_COL   = "Name"
DEV_ALIAS_COL  = "Alias"
DEV_LONG_COL   = "Device_Long_name"
DEV_TYPE_COL   = "Device_Type"
DEV_OWNER_COL  = "business_owner"

def _dvc(name):
    return F.col(f"`{name}`") if name in dev.columns else F.lit(None).cast("string")

dev_enriched = (
    dev
    .withColumn(
        "enrichment_text",
            F.concat_ws(
                "\n",
                F.concat(F.lit("Device Name: "),  F.coalesce(_dvc(DEV_NAME_COL), F.lit(""))),
                F.concat(F.lit("Device Type: "),  F.coalesce(_dvc(DEV_TYPE_COL), F.lit(""))),
                F.concat(F.lit("Business Owner: "), expand_business_owner_udf(_dvc(DEV_OWNER_COL))),
            ),
    )
    .withColumn("canonical_id",
                F.coalesce(_dvc(DEV_LONG_COL), _dvc(DEV_NAME_COL)))
)

ref_device = (
    dev_enriched
    .withColumn("key_norm", F.explode(
        device_variants_udf(_dvc(DEV_NAME_COL), _dvc(DEV_ALIAS_COL), _dvc(DEV_LONG_COL))))
    .filter(F.col("key_norm").isNotNull() & (F.length("key_norm") >= 4))
    .withColumn("entity_type", F.lit("DEVICE"))
    .withColumn("enrichment_text", F.concat(
        F.lit("[DEVICE] \""), F.col("key_norm"), F.lit("\"\n"),
        F.lit("Canonical: "), F.col("canonical_id"), F.lit("\n"),
        F.col("enrichment_text")
    ))
    .select(*LOOKUP_SCHEMA)
    .dropDuplicates(["key_norm"])
)

(ref_device.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_DEVICE))
spark.sql(f"COMMENT ON TABLE {REF_DEVICE} IS "
          "'Device lookup: normalized name/alias/long_name variants matched as substrings; "
          "context = Device_Type + business_owner (abbreviations expanded).'")

# read back for the substring extractor
device_key_set = set(r["key_norm"] for r in spark.table(REF_DEVICE).select("key_norm").collect())
extract_device_exact_udf = make_extract_device_exact_udf(device_key_set)
print(f"Device known-set size: {len(device_key_set):,}")

display(spark.table(REF_DEVICE).limit(20))

In [0]:
# ============================================================================
# CELL 7c — REF_VENDORS  (substring name pool; context = vendor name + country)
# ============================================================================
vnd = spark.table(RAW_VENDORS)
print("RAW_VENDORS columns:", vnd.columns)

VND_NM_COL      = "nm"
VND_COUNTRY_COL = "country_name"
VND_COUNT_COL   = "vendor_count"

def _vc(name):
    return F.col(f"`{name}`") if name in vnd.columns else F.lit(None).cast("string")

vnd_enriched = (
    vnd
    .filter(F.col(VND_COUNT_COL) >= 5)           # drop long-tail one-off vendors; tune as needed
    .withColumn(
        "enrichment_text",
        F.concat_ws(
            "\n",
            F.concat(F.lit("Vendor (External Supplier): "), F.coalesce(_vc(VND_NM_COL), F.lit(""))),
            F.concat(F.lit("Country: "), F.coalesce(_vc(VND_COUNTRY_COL), F.lit(""))),
        ),
    )
    .withColumn("canonical_id", _vc(VND_NM_COL))
)

ref_vendors = (
    vnd_enriched
    .withColumn("key_norm", F.explode(vendor_variants_udf(_vc(VND_NM_COL))))
    .filter(F.col("key_norm").isNotNull() & (F.length("key_norm") >= 4))
    .withColumn("entity_type", F.lit("VENDOR"))
    .withColumn("enrichment_text", F.concat(
        F.lit("[VENDOR] \""), F.col("key_norm"), F.lit("\"\n"),
        F.lit("Canonical: "), F.col("canonical_id"), F.lit("\n"),
        F.col("enrichment_text")
    ))
    .select(*LOOKUP_SCHEMA)
    .dropDuplicates(["key_norm"])
)

(ref_vendors.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_VENDORS))
spark.sql(f"COMMENT ON TABLE {REF_VENDORS} IS "
          "'Vendor lookup: normalized vendor name matched as substring; "
          "context = vendor name + country. vendor_count >= 5 threshold applied.'")

# Vendor extraction uses a broadcast JOIN on n-grams in the src_keys cell — no factory UDF needed.
print(f"Vendor pool size (post-filter): {spark.table(REF_VENDORS).count():,}")

display(spark.table(REF_VENDORS).limit(20))

In [0]:
# ============================================================================
# CELL 8 — REF_UNIFIED  (union of all lookups; single table to join against)
# ============================================================================
ref_unified = (
    spark.table(REF_CLINICAL).select(*LOOKUP_SCHEMA)
    .unionByName(spark.table(REF_DOCS).select(*LOOKUP_SCHEMA))
    .unionByName(spark.table(REF_ACRONYM).select(*LOOKUP_SCHEMA))
    .unionByName(spark.table(REF_CRO).select(*LOOKUP_SCHEMA))
    .unionByName(spark.table(REF_DEVICE).select(*LOOKUP_SCHEMA))
    .unionByName(spark.table(REF_VENDORS).select(*LOOKUP_SCHEMA))
    .dropDuplicates(["key_norm", "entity_type"])
)


(ref_unified.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_UNIFIED))
spark.sql(f"COMMENT ON TABLE {REF_UNIFIED} IS "
          "'Unified glossary: clinical + document + acronym + CRO + device + vendor lookups, one row per (key_norm, entity_type).'")

print("Row counts per entity_type:")
display(spark.table(REF_UNIFIED).groupBy("entity_type").count())
display(spark.table(REF_UNIFIED).limit(20))

In [0]:
# ============================================================================
# CELL 9 — SOURCE RESOLUTION  (COMPRESS TO EVENT GRAIN FIRST)
# Builds deviation_embed_input from EXACT reference matches. 
# ============================================================================
src = spark.table(SOURCE_TABLE)
print("SOURCE columns:", src.columns)

SRC_ID_COL       = "Event_Number"
SRC_PROTOCOL_COL = "Study_Protocol"
SRC_PROGRAM_COL  = "Program_Number"
SRC_DOC_COL      = "Document_or_Process"

SRC_CORE_COLS   = ["Event_Title", "Event_Description"]
SRC_EVENT_COLS  = ["Impact_Assessment", "Quality_Final_Assessment",
                   "Root_Cause_Category", "Root_Cause_SubCategory"]
SRC_ROWGRAIN_COLS = ["Action_Text"]
SRC_MID_COLS = SRC_CORE_COLS + SRC_EVENT_COLS   # mid tier = core + event (no action)


def _sc(name):
    return F.col(f"`{name}`") if name in src.columns else F.lit(None).cast("string")

# ---- 1. COMPRESS to one row per Event_Number ----
agg_exprs = []
# event-stable columns -> first non-null
for c in SRC_CORE_COLS + SRC_EVENT_COLS + [SRC_PROTOCOL_COL, SRC_PROGRAM_COL, SRC_DOC_COL]:
    if c in src.columns:
        agg_exprs.append(F.first(_sc(c), ignorenulls=True).alias(c))
# row-grain columns -> distinct NON-NULL values joined into one string
for c in SRC_ROWGRAIN_COLS:
    if c in src.columns:
        # collect_list already drops nulls; array_join avoids empty-string artifacts
        agg_exprs.append(
            F.array_join(F.array_distinct(F.collect_list(_sc(c))), "\n").alias(c)
        )

src_event = (
    src.groupBy(F.col(f"`{SRC_ID_COL}`").alias("pr_id"))
       .agg(*agg_exprs)
)
src.groupBy(F.col(f"`{SRC_ID_COL}`").cast("string").alias("pr_id"))
print(f"Compressed source: {src.count():,} rows -> {src_event.count():,} events")

# ---- 2. Clean + English-filter the event-grain columns ----
def _ec(name):
    return F.col(f"`{name}`") if name in src_event.columns else F.lit("").cast("string")

def _clean_english_col(name):
    return keep_if_english_udf(clean_freetext_udf(_ec(name)))

FULL_TEXT_COLS = SRC_CORE_COLS + SRC_EVENT_COLS + SRC_ROWGRAIN_COLS  # combined_text scope

src_base = (
    src_event.select(
        "pr_id",
        _ec(SRC_PROTOCOL_COL).alias("study_protocol"),
        _ec(SRC_PROGRAM_COL).alias("program_number"),
        _ec(SRC_DOC_COL).alias("document_or_process"),
        *[_clean_english_col(c).alias(f"__en_{c}") for c in FULL_TEXT_COLS],
        # raw values alongside English-filtered ones — needed to detect non-English content
        *[F.coalesce(_ec(c), F.lit("")).alias(f"__raw_{c}") for c in FULL_TEXT_COLS],
    )
    # Drop rows where ANY fine-embedding column has content but failed English detection.
    # A column is "non-English" when the raw value is non-empty but keep_if_english returned null.
    # e.g. an all-Japanese event like 5588310: Event_Title has text but __en_Event_Title is null.
    .filter(
        F.greatest(*[
            F.when(
                (F.length(F.trim(F.col(f"__raw_{c}"))) > 0) & F.col(f"__en_{c}").isNull(),
                F.lit(1)
            ).otherwise(F.lit(0))
            for c in FULL_TEXT_COLS
        ]) == 0
    )
    .drop(*[f"__raw_{c}" for c in FULL_TEXT_COLS])  # clean up helper cols
    .withColumn(
        "combined_text",   # full context: core + event + action_text
        F.concat_ws("\n",
            *[F.concat(F.lit(f"{c}: "), F.coalesce(F.col(f"__en_{c}"), F.lit("")))
              for c in FULL_TEXT_COLS]),
    )
    .withColumn(
        "core_text",       # title + description only
        F.concat_ws("\n",
            *[F.concat(F.lit(f"{c}: "), F.coalesce(F.col(f"__en_{c}"), F.lit("")))
              for c in SRC_CORE_COLS]),
    )
    .withColumn(
        "mid_text",        # core + impact + quality + root-cause (no action)
        F.concat_ws("\n",
            *[F.concat(F.lit(f"{c}: "), F.coalesce(F.col(f"__en_{c}"), F.lit("")))
              for c in SRC_MID_COLS]),
    )
    .select("pr_id", "combined_text", "mid_text", "core_text",
            "study_protocol", "program_number", "document_or_process")
    .filter(F.length(F.trim(F.col("combined_text"))) > 0)
)

# --- Extract all entity keys (free text + structured columns) ---
src_keys = (
    src_base
    .withColumn("clinical_keys",   extract_clinical_keys_udf(F.col("combined_text")))
    .withColumn("protocol_keys",   protocol_keys_udf(F.col("study_protocol")))
    .withColumn("program_keys",    program_keys_udf(F.col("program_number")))
    .withColumn("doc_keys",        extract_doc_keys_udf(F.col("combined_text")))          # free text
    .withColumn("doc_struct_keys", extract_doc_keys_udf(F.col("document_or_process")))    # structured
    .withColumn("acr_keys",        extract_acronym_keys_udf(F.col("combined_text")))
    .withColumn("device_keys",     extract_device_exact_udf(F.col("combined_text")))
)

# ---- Vendor extraction via broadcast JOIN on normalised text n-grams ----
VENDOR_STOPWORDS = frozenset({"patient", "patients"})
ref_vnd_keys = F.broadcast(
    spark.table(REF_VENDORS)
    .select(F.col("key_norm").alias("vnd_key"))
    .filter(F.size(F.split(F.col("vnd_key"), " ")) >= 2)       # require >=2 words
    .filter(~F.col("vnd_key").rlike(r'^patients?\b'))           # exclude patient/patients even in multi-word names
)
vendor_exploded = (
    src_base
    .select("pr_id", F.explode(text_ngrams_udf(F.col("combined_text"))).alias("ngram"))
    .join(ref_vnd_keys, F.col("ngram") == F.col("vnd_key"), "inner")
    .select("pr_id", F.col("vnd_key").alias("key_norm"))
    .withColumn("entity_type", F.lit("VENDOR"))
    .dropDuplicates(["pr_id", "key_norm"])
)

# ---- CRO extraction via broadcast JOIN on normalised text n-grams ----
ref_cro_keys = F.broadcast(
    spark.table(REF_CRO).select(F.col("key_norm").alias("cro_key"))
)
cro_exploded = (
    src_base
    .select("pr_id", F.explode(text_ngrams_udf(F.col("combined_text"))).alias("ngram"))
    .join(ref_cro_keys, F.col("ngram") == F.col("cro_key"), "inner")
    .select("pr_id", F.col("cro_key").alias("key_norm"))
    .withColumn("entity_type", F.lit("CRO"))
    .dropDuplicates(["pr_id", "key_norm"])
)

def _explode_keys(df, arr_col, etype):
    return (df.select("pr_id", F.explode(F.col(arr_col)).alias("key_norm"))
              .filter(F.col("key_norm").isNotNull() & (F.col("key_norm") != ""))
              .withColumn("entity_type", F.lit(etype)))

exploded = (
    _explode_keys(src_keys, "clinical_keys",   "CLINICAL_ID")
    .unionByName(_explode_keys(src_keys, "protocol_keys",   "CLINICAL_ID"))
    .unionByName(_explode_keys(src_keys, "program_keys",    "CLINICAL_ID"))
    .unionByName(_explode_keys(src_keys, "doc_keys",        "DOCUMENT"))    # free text
    .unionByName(_explode_keys(src_keys, "doc_struct_keys", "DOCUMENT"))    # structured
    .unionByName(_explode_keys(src_keys, "acr_keys",        "ACRONYM"))
    .unionByName(cro_exploded)
    .unionByName(_explode_keys(src_keys, "device_keys",     "DEVICE"))
    .unionByName(vendor_exploded)
    .dropDuplicates(["pr_id", "key_norm", "entity_type"])
)

ref = spark.table(REF_UNIFIED).select(
    "key_norm", "entity_type",
    F.col("canonical_id").alias("ref_canonical_id"),
    F.col("enrichment_text").alias("ref_enrichment"),
)

resolved = exploded.join(ref, on=["key_norm", "entity_type"], how="left")

enrichment_per_pr = (
    resolved.filter(F.col("ref_enrichment").isNotNull())
    .groupBy("pr_id")
    .agg(F.concat_ws("\n---\n", F.collect_set("ref_enrichment")).alias("injected_context"))
)

def _with_ctx(text_col):
    return F.when(
        F.length(F.trim("injected_context")) > 0,
        F.concat_ws("\n\n", F.col(text_col),
            F.concat(F.lit("=== RESOLVED REFERENCES ===\n"), F.col("injected_context"))),
    ).otherwise(F.col(text_col))

embed_ready = (
    src_base.join(enrichment_per_pr, on="pr_id", how="left")
    .withColumn("injected_context", F.coalesce(F.col("injected_context"), F.lit("")))
    .withColumn("embedding_text",      _with_ctx("combined_text"))
    .withColumn("core_embedding_text", _with_ctx("core_text"))
    .withColumn("mid_embedding_text",  _with_ctx("mid_text"))
    .select("pr_id", "combined_text", "core_text", "mid_text",
            "injected_context", "embedding_text", "core_embedding_text", "mid_embedding_text")
)

# ---- Write via staging (safe on reruns where embed_input already exists) ----
EMBED_INPUT = f"{CATALOG}.{ALYT}.deviation_embed_input"
STAGING     = f"{CATALOG}.{ALYT}.deviation_embed_input_staging"

(embed_ready.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(STAGING))
(spark.table(STAGING).write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(EMBED_INPUT))
spark.sql(f"DROP TABLE IF EXISTS {STAGING}")

print("Coverage — deviations with >=1 resolved reference:")
display(embed_ready.select(
    F.count("*").alias("total"),
    F.sum(F.when(F.length("injected_context") > 0, 1).otherwise(0)).alias("with_context"),
))
display(spark.table(EMBED_INPUT).limit(10))

In [0]:
# DIAGNOSTIC — how many source rows extract each entity type, and how many join
print("Source rows with ≥1 extracted key (pre-join, by entity type):")
display(
    exploded.groupBy("entity_type")
    .agg(F.countDistinct("pr_id").alias("rows_with_key"))
    .orderBy("entity_type")
)

print("Exploded keys that actually matched REF_UNIFIED:")
display(
    exploded.join(ref, on=["key_norm", "entity_type"], how="left")
    .groupBy("entity_type")
    .agg(
        F.count("*").alias("extracted"),
        F.sum(F.when(F.col("ref_enrichment").isNotNull(), 1).otherwise(0)).alias("matched"),
    )
)

print("Top extracted keys that FAILED to match (fix these first):")
display(
    exploded.join(ref, on=["key_norm", "entity_type"], how="left")
    .filter(F.col("ref_enrichment").isNull())
    .groupBy("entity_type", "key_norm").count()
    .orderBy(F.col("count").desc()).limit(30)
)

In [0]:
# ============================================================================
# SECTION I — BGE-M3 EMBEDDINGS TABLE  (three tiers)  [DEVIATIONS, not glossary]
#
# Embeds the ENRICHED DEVIATIONS produced by Cell 9/10 (deviation_embed_input),
# whose text columns already contain the injected lexical/symbolic context.
#
#   embedding      (fine) — embedding_text      : combined_text + resolved refs
#   mid_embedding  (mid)  — mid_embedding_text   : core+event text + resolved refs
#   core_embedding (core) — core_embedding_text  : title+description + resolved refs
#
# Keyed on pr_id (Event_Number). One BGE-M3 pass over [fine|mid|core] → split.
#
# MODEL: BAAI/bge-m3  |  backbone XLM-RoBERTa-large  |  max supported len = 8192
#        → MAX_LEN = 3000 is well within range (covers ~99%+ of enriched rows).
# ============================================================================
import time
import numpy as np
from pyspark.sql import functions as F, types as T
from FlagEmbedding import BGEM3FlagModel

EMBED_INPUT     = f"{CATALOG}.{ALYT}.deviation_embed_input"   # ← Cell 9/10 output
EMB_TABLE       = f"{CATALOG}.{ALYT}.deviation_embeddings"
EMB_INDEX_FINE  = f"{CATALOG}.{ALYT}.deviation_emb_fine_idx"
EMB_INDEX_MID   = f"{CATALOG}.{ALYT}.deviation_emb_mid_idx"
EMB_INDEX_CORE  = f"{CATALOG}.{ALYT}.deviation_emb_core_idx"
EMB_DIM         = 1024
MAX_LEN         = 8192         # ← raised from 640; BGE-M3 supports up to 8192
BATCH           = 64           # ← lowered from 256; long sequences need smaller batches
VS_ENDPOINT     = "deviation-retrieval-vs"

# ── 1. Load BGE-M3 on GPU driver (FP16 for H100) ──────────────────────────
print("Loading BAAI/bge-m3 (first run downloads ~3 GB from HuggingFace) …")
bge = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("Model loaded.")

# ── 2. Pull ENRICHED DEVIATIONS to driver ─────────────────────────────────
src_pd = (
    spark.table(EMBED_INPUT)
    .select(
        F.col("pr_id"),
        "combined_text", "mid_text", "core_text",
        "injected_context",
        "embedding_text", "mid_embedding_text", "core_embedding_text",
    )
    .toPandas()
)
src_pd["pr_id"] = src_pd["pr_id"].astype(str)

# Safety: never feed None to the encoder
for c in ["embedding_text", "mid_embedding_text", "core_embedding_text",
          "combined_text", "mid_text", "core_text", "injected_context"]:
    src_pd[c] = src_pd[c].fillna("")

n = len(src_pd)
print(f"Rows: {n:,}  |  Total encode calls: {n * 3:,} (3 tiers in one pass)")

# ── 3. Single encode pass — all three text lists concatenated ─────────────
# Concatenate [fine | mid | core], encode once, then split by n.
all_texts = (
    src_pd["embedding_text"].tolist()
    + src_pd["mid_embedding_text"].tolist()
    + src_pd["core_embedding_text"].tolist()
)

# ---- Length-sort batching -------------------------------------------------
# At MAX_LEN=3000, padding is per-batch: one long row pads its whole batch up.
# Sort by token-ish length so long rows batch together and short rows stay
# short — huge padding savings since ~half the rows are < 640 tokens.
# We approximate length by tokenizer output for accurate ordering.
tok = bge.tokenizer
approx_len = [len(tok.encode(t, add_special_tokens=True)) for t in all_texts]
order      = sorted(range(len(all_texts)), key=lambda i: approx_len[i])
inv        = np.argsort(order)                       # to restore original order
all_texts_sorted = [all_texts[i] for i in order]

all_vecs_sorted = []
t0 = time.time()
for i in range(0, len(all_texts_sorted), BATCH):
    chunk = all_texts_sorted[i : i + BATCH]
    out = bge.encode(
        chunk,
        batch_size=len(chunk),
        max_length=MAX_LEN,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    all_vecs_sorted.extend(out["dense_vecs"].tolist())

# Restore original ordering
all_vecs = [all_vecs_sorted[inv[i]] for i in range(len(all_texts))]

print(f"Encoded {len(all_vecs):,} vectors in {time.time()-t0:.1f}s  |  dim={len(all_vecs[0])}")
assert len(all_vecs[0]) == EMB_DIM

# Split back into three tiers
src_pd["embedding"]      = all_vecs[:n]
src_pd["mid_embedding"]  = all_vecs[n : 2 * n]
src_pd["core_embedding"] = all_vecs[2 * n :]

# ── 4. Write Delta table with CDF + PRIMARY KEY (pr_id) ─────────────────────
emb_schema = T.StructType([
    T.StructField("pr_id",               T.StringType(),             False),  # ← PK
    T.StructField("combined_text",        T.StringType(),             True),
    T.StructField("mid_text",             T.StringType(),             True),
    T.StructField("core_text",            T.StringType(),             True),
    T.StructField("injected_context",     T.StringType(),             True),
    T.StructField("embedding_text",       T.StringType(),             True),
    T.StructField("mid_embedding_text",   T.StringType(),             True),
    T.StructField("core_embedding_text",  T.StringType(),             True),
    T.StructField("embedding",            T.ArrayType(T.FloatType()), True),
    T.StructField("mid_embedding",        T.ArrayType(T.FloatType()), True),
    T.StructField("core_embedding",       T.ArrayType(T.FloatType()), True),
])

# Write in chunks to stay under Spark Connect's 3GB local relation limit
CHUNK_SIZE = 50_000
for i in range(0, len(src_pd), CHUNK_SIZE):
    chunk_pdf = src_pd.iloc[i : i + CHUNK_SIZE]
    chunk_df = spark.createDataFrame(chunk_pdf, schema=emb_schema)
    write_mode = "overwrite" if i == 0 else "append"
    (chunk_df.write
        .mode(write_mode)
        .option("overwriteSchema", "true" if i == 0 else "false")
        .saveAsTable(EMB_TABLE))
    print(f"  Written chunk {i // CHUNK_SIZE + 1}: rows {i}–{min(i + CHUNK_SIZE, len(src_pd)) - 1}")

spark.sql(f"""
    ALTER TABLE {EMB_TABLE}
    SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

try:
    spark.sql(f"ALTER TABLE {EMB_TABLE} ALTER COLUMN pr_id SET NOT NULL")
    spark.sql(f"ALTER TABLE {EMB_TABLE} ADD CONSTRAINT pk_pr_id PRIMARY KEY (pr_id)")
    print("PRIMARY KEY constraint declared.")
except Exception as e:
    if "already exists" in str(e).lower():
        print("PRIMARY KEY constraint already exists — skipping.")
    else:
        raise

row_count = spark.table(EMB_TABLE).count()
print(f"\n✓ {EMB_TABLE}")
print(f"  Rows: {row_count:,}  |  dim: {EMB_DIM}  |  CDF: on  |  PK: pr_id")
print(f"  MAX_LEN: {MAX_LEN}  |  BATCH: {BATCH}  |  length-sorted batching: on")
print(f"  Columns: embedding (fine) | mid_embedding | core_embedding")

In [0]:
# ============================================================================
# SECTION I-b — Sentence-Transformers (GTE-Large-en v1.5) on core_embedding_text
#
# Comparison embedding alongside BGE-M3. Uses Alibaba's GTE-Large-en-v1.5:
#   - 1024-dim (same as BGE-M3 → direct comparison)
#   - 8192 max tokens (same as BGE-M3 → no truncation)
#   - Trained on diverse English text with bi-directional attention
#
# Uses src_pd from the preceding BGE-M3 cell (still in driver memory).
# ============================================================================
from sentence_transformers import SentenceTransformer
import time
import numpy as np
from pyspark.sql import types as T

# ---- Load model on GPU ----
S2V_MODEL_NAME = "Alibaba-NLP/gte-large-en-v1.5"
s2v_model = SentenceTransformer(S2V_MODEL_NAME, trust_remote_code=True, device="cuda")
s2v_model.max_seq_length = 8192  # use full context window
S2V_DIM = s2v_model.get_sentence_embedding_dimension()
print(f"Loaded: {S2V_MODEL_NAME}")
print(f"  dim={S2V_DIM}  |  max_seq_len={s2v_model.max_seq_length}")

# ---- Load data directly (self-contained — no dependency on earlier cells) ----
CATALOG = "us_gmsgq_dev"
ALYT    = "gms_us_alyt"
EMBED_INPUT = f"{CATALOG}.{ALYT}.deviation_embed_input"
src_pd = spark.table(EMBED_INPUT).select("pr_id", "core_embedding_text").toPandas()
src_pd["pr_id"] = src_pd["pr_id"].astype(str)
src_pd["core_embedding_text"] = src_pd["core_embedding_text"].fillna("")
print(f"Loaded {len(src_pd):,} rows from {EMBED_INPUT}")

# ---- Encode core_embedding_text ----
core_texts = src_pd["core_embedding_text"].tolist()

t0 = time.time()
s2v_vectors = s2v_model.encode(
    core_texts,
    batch_size=32,             # smaller batch for 8192-token sequences on GPU
    show_progress_bar=True,
    normalize_embeddings=True,  # L2-normalised for cosine similarity
)
print(f"Encoded {len(s2v_vectors):,} texts in {time.time()-t0:.1f}s  |  shape={s2v_vectors.shape}")

# ---- Add core_embedding_s2v as a new column to deviation_embeddings ----
EMB_TABLE = f"{CATALOG}.{ALYT}.deviation_embeddings"

# Build a small DataFrame with pr_id + the new vector column
s2v_pd_out = src_pd[["pr_id"]].copy()
s2v_pd_out["core_embedding_s2v"] = s2v_vectors.tolist()

s2v_sdf = spark.createDataFrame(
    s2v_pd_out,
    schema=T.StructType([
        T.StructField("pr_id",              T.StringType(),             False),
        T.StructField("core_embedding_s2v", T.ArrayType(T.FloatType()), True),
    ])
)

# Read existing embeddings table, join the new column, and overwrite
existing = spark.table(EMB_TABLE)
merged = existing.drop("core_embedding_s2v").join(s2v_sdf, on="pr_id", how="left")

(merged.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(EMB_TABLE))

print(f"\n\u2713 Added core_embedding_s2v to {EMB_TABLE}")
print(f"  Model: {S2V_MODEL_NAME}  |  dim: {S2V_DIM}")
print(f"  Text source: core_embedding_text (title + description + resolved refs)")
print(f"  Columns now: {spark.table(EMB_TABLE).columns}")

In [0]:
# ============================================================================
# SECTION J — AI SEARCH ENDPOINT + THREE DELTA SYNC INDEXES   [AISearchClient]
#
# Migrated from databricks.sdk (w.vector_search_*) to the canonical
# databricks-ai-search SDK (AISearchClient).  Uses your PRECOMPUTED BGE-M3
# vectors (embedding_vector_column + embedding_dimension) — Databricks does
# NOT recompute embeddings.
#
# One index per vector column (Delta Sync requires this):
#   embedding      → fine index
#   mid_embedding  → mid  index
#   core_embedding → core index
#
# Prereqs: EMB_TABLE has CDF enabled + a PRIMARY KEY on pr_id (Cell 13).
# ============================================================================
import time
import sys, os, databricks
# Extend databricks namespace to find notebook-scoped ai_search install
for _sp in sys.path:
    _dbp = os.path.join(_sp, "databricks")
    if os.path.isdir(os.path.join(_dbp, "ai_search")) and _dbp not in databricks.__path__:
        databricks.__path__.append(_dbp)
        break
from databricks.ai_search.client import AISearchClient

client = AISearchClient()          # auto-detects notebook credentials

# (Re)state names in case Python was restarted by the SDK install cell.
EMB_TABLE       = f"{CATALOG}.{ALYT}.deviation_embeddings"
EMB_INDEX_FINE  = f"{CATALOG}.{ALYT}.deviation_emb_fine_idx"
EMB_INDEX_MID   = f"{CATALOG}.{ALYT}.deviation_emb_mid_idx"
EMB_INDEX_CORE  = f"{CATALOG}.{ALYT}.deviation_emb_core_idx"
EMB_DIM         = 1024
VS_ENDPOINT     = "deviation-retrieval-vs"

# Metadata columns to return at query time (kept small; vectors excluded).
META_COLS = ["pr_id", "combined_text", "mid_text", "core_text", "injected_context"]


# ── helper: read a normalized state string out of index.describe() ─────────
def _describe_state(desc: dict) -> str:
    """describe() returns a plain dict; probe its status defensively."""
    st = (desc or {}).get("status", {}) or {}
    return str(
        st.get("detailed_state")
        or ("ONLINE_READY" if st.get("ready") is True else None)
        or st.get("message")
        or "UNKNOWN"
    )

def _describe_rows(desc: dict):
    st = (desc or {}).get("status", {}) or {}
    return st.get("indexed_row_count", "n/a")


# ── 1. Create (or reuse) the endpoint ──────────────────────────────────────
try:
    client.create_endpoint(name=VS_ENDPOINT, endpoint_type="STANDARD")
    print(f"Creating endpoint '{VS_ENDPOINT}' …")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Endpoint '{VS_ENDPOINT}' already exists — reusing.")
    else:
        raise

# ── 2. Create-or-reuse one Delta Sync index per vector column ──────────────
def _ensure_index(index_name: str, vector_col: str) -> None:
    try:
        kwargs = dict(
            endpoint_name=VS_ENDPOINT,
            index_name=index_name,
            source_table_name=EMB_TABLE,
            primary_key="pr_id",
            pipeline_type="TRIGGERED",
            embedding_dimension=EMB_DIM,          # existing-embeddings path
            embedding_vector_column=vector_col,   # our precomputed BGE-M3 column
        )
        try:
            client.create_delta_sync_index(columns_to_sync=META_COLS, **kwargs)
        except TypeError:
            # Older/newer SDK may not expose columns_to_sync → sync all columns.
            client.create_delta_sync_index(**kwargs)
        print(f"  Created : {index_name}  ({vector_col})  — self-syncs on init")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"  Exists  : {index_name}  ({vector_col})")
        else:
            raise

    # Poll describe() until ONLINE
    for attempt in range(60):
        try:
            desc = client.get_index(endpoint_name=VS_ENDPOINT,
                                    index_name=index_name).describe()
        except Exception as e:
            print(f"    [{attempt+1:02d}] describe() not ready: {e}")
            time.sleep(30); continue
        state = _describe_state(desc)
        print(f"    [{attempt+1:02d}] {state}")
        if "ONLINE" in state:
            return
        if "FAILED" in state:
            raise RuntimeError(f"Index '{index_name}' entered FAILED: {desc.get('status')}")
        time.sleep(30)
    raise TimeoutError(f"Index '{index_name}' did not reach ONLINE.")


print("\n── Fine index (embedding — combined_text + resolved refs) ──")
_ensure_index(EMB_INDEX_FINE, "embedding")

print("\n── Mid index (mid_embedding — core+event text + resolved refs) ──")
_ensure_index(EMB_INDEX_MID, "mid_embedding")

print("\n── Core index (core_embedding — title+description + resolved refs) ──")
_ensure_index(EMB_INDEX_CORE, "core_embedding")

print(f"\n✓ AI Search stack ready")
print(f"  Endpoint : {VS_ENDPOINT}")
print(f"  Fine idx : {EMB_INDEX_FINE}")
print(f"  Mid  idx : {EMB_INDEX_MID}")
print(f"  Core idx : {EMB_INDEX_CORE}")


In [0]:
# ============================================================================
# SECTION K — SIMILARITY SEARCH   [AISearchClient]  (per-tier, deviations) experiment 
#
# Uses index.similarity_search(query_vector=..., columns=..., num_results=...).
# Embed the query with the SAME BGE-M3 params used at index time (max_length).
#
# NOTE: if Python was restarted by the SDK-install cell, `bge` is gone.
#       The reload block below brings it back (model load only — no re-encode).
# ============================================================================
from databricks.ai_search.client import AISearchClient

client = AISearchClient()

# --- reload bge if it was wiped by restartPython() ---
try:
    bge  # noqa: F821  — still in memory?
except NameError:
    from FlagEmbedding import BGEM3FlagModel
    print("Reloading BAAI/bge-m3 …")
    bge = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
    print("Model reloaded.")

MAX_LEN     = 1024
QUERY       = "site inadvertently disclosed treatment assignment, possible unblinding"
NUM_RESULTS = 5

def _embed(text: str) -> list[float]:
    out = bge.encode(
        [text], batch_size=1, max_length=MAX_LEN,
        return_dense=True, return_sparse=False, return_colbert_vecs=False,
    )
    return out["dense_vecs"][0].tolist()

def _search(index_name: str, query_vec: list[float], label: str) -> None:
    index = client.get_index(endpoint_name=VS_ENDPOINT, index_name=index_name)
    res = index.similarity_search(
        query_vector=query_vec,
        columns=["pr_id", "combined_text", "injected_context"],
        num_results=NUM_RESULTS,
    )
    # similarity_search returns a dict: result.data_array = [[col..., score], ...]
    rows = res.get("result", {}).get("data_array", []) if isinstance(res, dict) else []
    print(f"── {label} ──  query: '{QUERY}'")
    for row in rows:
        pr_id, combined_text, injected_context, score = row
        snippet = (combined_text or "").replace("\n", " ")[:120]
        print(f"  {score:.4f}  [pr_id={pr_id}]  {snippet}")
        if injected_context:
            print(f"           refs: {injected_context.replace(chr(10), ' ')[:120]}")
    print()

q_vec = _embed(QUERY)
_search(EMB_INDEX_FINE, q_vec, "FINE  (combined_text + resolved refs)")
_search(EMB_INDEX_MID,  q_vec, "MID   (core+event text + resolved refs)")
_search(EMB_INDEX_CORE, q_vec, "CORE  (title+description + resolved refs)")


In [0]:
# ============================================================================
# DIAGNOSTICS   [AISearchClient]  — embedding table + AI Search index status
# ============================================================================
from databricks.ai_search.client import AISearchClient

client = AISearchClient()

def _describe_state(desc: dict) -> str:
    st = (desc or {}).get("status", {}) or {}
    return str(
        st.get("detailed_state")
        or ("ONLINE_READY" if st.get("ready") is True else None)
        or st.get("message") or "UNKNOWN"
    )

def _describe_rows(desc: dict):
    st = (desc or {}).get("status", {}) or {}
    return st.get("indexed_row_count", "n/a")

# ── 1. Embedding table: rows, PK integrity, dims, enrichment coverage ──────
try:
    df = spark.table(EMB_TABLE)
    row_count = df.count()
    sample = df.select(
        F.size("embedding").alias("fine_dim"),
        F.size("mid_embedding").alias("mid_dim"),
        F.size("core_embedding").alias("core_dim"),
    ).first()
    cov = df.select(
        F.count("*").alias("total"),
        F.sum(F.when(F.length(F.trim("injected_context")) > 0, 1).otherwise(0)).alias("with_ctx"),
        F.countDistinct("pr_id").alias("distinct_pr_id"),
    ).first()
    pct = (100.0 * cov["with_ctx"] / cov["total"]) if cov["total"] else 0.0

    print(f"✓ {EMB_TABLE}")
    print(f"  Rows            : {row_count:,}")
    print(f"  Distinct pr_id  : {cov['distinct_pr_id']:,}  (should == rows)")
    print(f"  dims (f/m/c)    : {sample['fine_dim']} / {sample['mid_dim']} / {sample['core_dim']}")
    print(f"  With context    : {cov['with_ctx']:,} / {cov['total']:,}  ({pct:.1f}%)")
    if cov["distinct_pr_id"] != row_count:
        print("  ⚠  Duplicate pr_id — PRIMARY KEY assumption violated.")
    if EMB_DIM not in (sample["fine_dim"], sample["mid_dim"], sample["core_dim"]):
        print(f"  ⚠  A tier's dim != {EMB_DIM}.")
    if pct == 0.0:
        print("  ⚠  No injected context — check Cell 10 enrichment join.")
except Exception as e:
    print(f"✗ Embedding table missing or empty: {e}")

# ── 2. AI Search index status ──────────────────────────────────────────────
print()
for label, idx_name in [("FINE", EMB_INDEX_FINE), ("MID", EMB_INDEX_MID), ("CORE", EMB_INDEX_CORE)]:
    try:
        desc = client.get_index(endpoint_name=VS_ENDPOINT, index_name=idx_name).describe()
        print(f"  [{label:4}]  {_describe_state(desc)}  |  indexed rows: {_describe_rows(desc)}")
    except Exception as e:
        print(f"  [{label:4}]  NOT FOUND — {e}")


In [0]:
# ============================================================================
# CELL 10.5 — TOKEN-LENGTH DIAGNOSTIC on deviation_embed_input
# Checks the three ENRICHED text columns (context already injected) against
# MAX_LEN to see if anything is being truncated by BGE-M3.
#   embedding_text       (fine)  = combined_text + resolved refs
#   mid_embedding_text   (mid)   = core+event text + resolved refs
#   core_embedding_text  (core)  = title+description + resolved refs
# ============================================================================
import numpy as np
import pandas as pd
from FlagEmbedding import BGEM3FlagModel

EMBED_INPUT = f"{CATALOG}.{ALYT}.deviation_embed_input"
MAX_LEN     = 640          # ← must match the value you use in the embedding cell

TEXT_COLS = ["embedding_text", "mid_embedding_text", "core_embedding_text"]

# ── 1. Load the model just for its tokenizer (skip if `bge` already in memory) ──
try:
    tok = bge.tokenizer                      # reuse if the embedding cell already ran
    print("Reusing already-loaded BGE-M3 tokenizer.")
except NameError:
    print("Loading BAAI/bge-m3 tokenizer …")
    bge = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
    tok = bge.tokenizer
    print("Tokenizer ready.")

# ── 2. Pull the enriched text columns to the driver ───────────────────────────
pdf = (
    spark.table(EMBED_INPUT)
    .select("pr_id", *TEXT_COLS)
    .toPandas()
)
for c in TEXT_COLS:
    pdf[c] = pdf[c].fillna("")
print(f"Rows loaded: {len(pdf):,}")

# ── 3. Count tokens per column (with special tokens, as the encoder sees them) ─
def token_len(text):
    # add_special_tokens=True -> includes [CLS]/[SEP], matching bge.encode()
    return len(tok.encode(text, add_special_tokens=True))

for c in TEXT_COLS:
    pdf[f"{c}__tok"] = pdf[c].map(token_len)

# ── 4. Report distribution + truncation exposure per tier ─────────────────────
rows = []
for c in TEXT_COLS:
    lengths = pdf[f"{c}__tok"].to_numpy()
    over = int((lengths > MAX_LEN).sum())
    rows.append({
        "column"       : c,
        "min"          : int(lengths.min()),
        "mean"         : round(float(lengths.mean()), 1),
        "median"       : int(np.percentile(lengths, 50)),
        "p95"          : int(np.percentile(lengths, 95)),
        "p99"          : int(np.percentile(lengths, 99)),
        "max"          : int(lengths.max()),
        f"rows_over_{MAX_LEN}" : over,
        "pct_truncated": round(100 * over / len(pdf), 2),
    })

summary = pd.DataFrame(rows).set_index("column")
print(f"\n=== Token-length summary (MAX_LEN = {MAX_LEN}) ===")
print(summary.to_string())

# ── 5. Show the worst offenders on the fine tier (most likely to clip) ─────────
FINE = "embedding_text"
worst = (
    pdf.sort_values(f"{FINE}__tok", ascending=False)
       .loc[:, ["pr_id", f"{FINE}__tok"]]
       .head(15)
       .rename(columns={f"{FINE}__tok": "fine_tokens"})
)
print(f"\n=== Top 15 longest '{FINE}' rows ===")
print(worst.to_string(index=False))

# Optional: keep a Spark/Delta copy of the token counts for later reference
# (spark.create

In [0]:
from FlagEmbedding import BGEM3FlagModel
lens = [len(bge.tokenizer.encode(t)) for t in src_pd["embedding_text"]]
import numpy as np
for p in [50, 90, 95, 99, 100]:
    print(f"p{p}: {np.percentile(lens, p):.0f} tokens")
pct_over_640 = np.mean([l > 640 for l in lens]) * 100
print(f"% over 640: {pct_over_640:.2f}%")